In [1]:
import pandas as pd
import networkx as nx
from sklearn.model_selection import GroupShuffleSplit

# 1. Load your dataset
df = pd.read_csv("../preprocessing/train_data_with_preprocessing2.csv")

# 2. Build the Bipartite Graph
G = nx.Graph()

# Add prefixes to avoid collisions between identical context and reply strings
df['ctx_node'] = "CTX_" + df['prev_clean'].astype(str)
df['rep_node'] = "REP_" + df['target_clean'].astype(str)

# Create edges based on rows
edges = list(zip(df['ctx_node'], df['rep_node']))
G.add_edges_from(edges)

# 3. Find Connected Components
components = list(nx.connected_components(G))

# Map every node to a unique Component ID
node_to_component = {}
for component_id, nodes in enumerate(components):
    for node in nodes:
        node_to_component[node] = component_id

# 4. Map the Component IDs back to the original dataframe
# (You only need to check the context node, as the reply node is guaranteed to have the same ID)
df['component_id'] = df['ctx_node'].map(node_to_component)

# 5. Split the data ensuring components are kept together
# GroupShuffleSplit ensures all rows with the same component_id stay in the same split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, val_idx = next(gss.split(df, groups=df['component_id']))

train_df = df.iloc[train_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])
val_df = df.iloc[val_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])

print(f"Training rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")

Training rows: 17991
Validation rows: 1440


In [2]:
train_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)
val_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

from sklearn.metrics import (
  accuracy_score,
  f1_score,
  classification_report,
  confusion_matrix
)

In [3]:
target_col = 'label'

In [4]:
# Extract text columns for TF-IDF
X_context_train = train_df['context'].fillna("").astype(str)
X_reply_train = train_df['reply'].fillna("").astype(str)
y_train = train_df[target_col].fillna("nan").astype(str)

X_context_dev = val_df['context'].fillna("").astype(str)
X_reply_dev = val_df['reply'].fillna("").astype(str)
y_dev = val_df[target_col].fillna("nan").astype(str)

In [5]:
# Word-level TF-IDF
context_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [6]:
# Character-level TF-IDF
context_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [7]:
# Fit only on training data, transform both
Cw_train = context_word.fit_transform(X_context_train)
Cw_dev = context_word.transform(X_context_dev)

Rw_train = reply_word.fit_transform(X_reply_train)
Rw_dev = reply_word.transform(X_reply_dev)

Cc_train = context_char.fit_transform(X_context_train)
Cc_dev = context_char.transform(X_context_dev)

Rc_train = reply_char.fit_transform(X_reply_train)
Rc_dev = reply_char.transform(X_reply_dev)

In [8]:
# Combine all four sparse feature matrices
X_train = hstack([Cw_train, Cc_train, Rw_train, Rc_train], format="csr")
X_dev = hstack([Cw_dev, Cc_dev, Rw_dev, Rc_dev], format="csr")

print("Training feature matrix:", X_train.shape)
print("Development feature matrix:", X_dev.shape)

Training feature matrix: (17991, 36109)
Development feature matrix: (1440, 36109)


In [9]:
model = SVC(
  C=1.0,
  kernel='linear',
  class_weight='balanced',
  gamma='scale',
)

model.fit(X_train, y_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [11]:
y_pred = model.predict(X_dev)

print("Accuracy:", accuracy_score(y_dev, y_pred))
print("F1 Score:", f1_score(y_dev, y_pred, average='weighted'))
print("Classification Report:\n", classification_report(y_dev, y_pred))

Accuracy: 0.4826388888888889
F1 Score: 0.39046985812992485
Classification Report:
                                      precision    recall  f1-score   support

                     Dehumanization       0.85      0.04      0.08       690
                         Harassment       1.00      1.00      1.00         2
                        Hate Speech       0.43      0.85      0.57        27
                    Identity Attack       1.00      0.98      0.99        95
 Incitement / Encouragement of Harm       0.00      0.00      0.00         2
                             Insult       0.30      0.88      0.45       130
                 Mockery / Ridicule       1.00      0.88      0.93        81
                          Profanity       0.47      0.28      0.35        25
Sexual Harassment / Objectification       0.52      0.38      0.43        40
                             Threat       0.98      0.90      0.94        50
      Toxic Endorsement / Agreement       1.00      1.00      1.00   

c:\Users\rajde\anaconda3\envs\pytrch310\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\rajde\anaconda3\envs\pytrch310\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\rajde\anaconda3\envs\pytrch310\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [13]:
df_test = pd.read_csv("../preprocessing/test_data_with_preprocessing2.csv")
df_test.head()

,context,reply,context_clean,reply_clean,label,label_id,is_toxic
0,HSC exam will be conducted on time in the enti...,right decision,hsc exam will be conducted on time in the enti...,right decision,NaN,0,0
1,HSC exam will be conducted on time in the enti...,You are absolutely right,hsc exam will be conducted on time in the enti...,you are absolutely right,NaN,0,0
2,Why was the test not delayed? Education Minist...,What is the name of the education minister who...,why was the test not delayed ? education minis...,what is the name of the education minister who...,Mockery / Ridicule,7,1
3,Why was the test not delayed? Education Minist...,What was asked and what was the answer?,why was the test not delayed ? education minis...,what was asked and what was the answer ?,NaN,0,0
4,Why was the test not delayed? Education Minist...,Helicopter meets Water meets,why was the test not delayed ? education minis...,helicopter meets water meets,Mockery / Ridicule,7,1


In [14]:
y_test = df_test[target_col].fillna("nan").astype(str)
X_test_context = df_test['context_clean'].fillna("").astype(str)
X_test_reply = df_test['reply_clean'].fillna("").astype(str)

In [16]:
Cw_test = context_word.transform(X_test_context)
Rw_test = reply_word.transform(X_test_reply)

Cc_test = context_char.transform(X_test_context)
Rc_test = reply_char.transform(X_test_reply)

In [17]:
X_test = hstack([Cw_test, Cc_test, Rw_test, Rc_test], format="csr")

In [18]:
print("Test Accuracy:", accuracy_score(y_test, model.predict(X_test)))

Test Accuracy: 0.4024390243902439


In [9]:
import optuna
import numpy as np
from sklearn.metrics import log_loss

optuna.logging.set_verbosity(optuna.logging.DEBUG)

def objective(trial):
  C = trial.suggest_float("C", 1e-3, 1e3, log=True)
  kernel = trial.suggest_categorical("kernel", ["linear", "poly", "rbf", "sigmoid"])
  gamma = trial.suggest_categorical("gamma", ["scale", "auto"])

  if kernel == "poly":
    degree = trial.suggest_int("degree", 2, 5)
  else:
    degree = 3  # Default value when not in use

  class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

  # 6
  model = SVC(
    C=C,
    kernel=kernel,
    degree=degree,
    gamma=gamma,
    class_weight=class_weight,
    random_state=42,
  )

  model.fit(X_train, y_train)
  # y_pred = model.predict(X_dev)
  # score = accuracy_score(y_dev, y_pred) 
  y_proba = model.predict_proba(X_dev)
  score = log_loss(y_dev, y_proba)  # Use log loss for evaluation

  return score

In [12]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))

[I 2026-11-02 22:29:15,965] A new study created in memory with name: no-name-643049d1-b907-46fc-a352-ccd3b089cada


In [13]:
study.optimize(objective, n_trials=50)

[W 2026-11-02 22:31:44,272] Trial 0 failed with parameters: {'C': 0.1767016940294795, 'kernel': 'linear', 'gamma': 'scale', 'class_weight': None} because of the following error: AttributeError("This 'SVC' has no attribute 'predict_proba'").
Traceback (most recent call last):
  File "c:\Users\rajde\anaconda3\envs\pytrch310\Lib\site-packages\sklearn\utils\_available_if.py", line 32, in _check
    check_result = self.check(obj)
                   ^^^^^^^^^^^^^^^
  File "c:\Users\rajde\anaconda3\envs\pytrch310\Lib\site-packages\sklearn\svm\_base.py", line 866, in _check_proba
    raise AttributeError(
AttributeError: predict_proba is not available when probability=False

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\rajde\anaconda3\envs\pytrch310\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\rajde\AppData\Lo

AttributeError: This 'SVC' has no attribute 'predict_proba'

# Added evaluation: binary toxicity and 12-class subtype

Uses the existing linear-kernel SVC configuration. It uses SVC, not a substituted LinearSVC. Kernel cache size only controls memory/performance.

The original cells and outputs above are unchanged. **Run this added section independently.** It reads the supplied preprocessing2 CSV pair and preserves the literal `None` label. Every model in this comparison fits its word/character TF-IDF vectorizers on **all 19,431 training rows**, then evaluates the same **1,804 test rows**. The feature settings and block order match the original hyperparameter notebooks. Separate classifiers learn binary `is_toxic` and 12-class `label_id` targets.

This is a fresh full-training refit, so these test scores can differ from older outputs based on a smaller training partition. No new hyperparameter search uses the test set. Rare-class performance requires macro F1 as well as accuracy. The test has already been examined during the course project, so these comparisons are exploratory.

Dependencies: `numpy pandas scipy scikit-learn`.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from IPython.display import display

for dual_folder in [Path.cwd(), *Path.cwd().parents]:
    if (dual_folder/'preprocessing/train_data_with_preprocessing2.csv').exists():
        dual_root=dual_folder
        break
else:
    raise FileNotFoundError('Open this notebook inside the project folder.')
dual_train_path=dual_root/'preprocessing/train_data_with_preprocessing2.csv'
dual_test_path=dual_root/'preprocessing/test_data_with_preprocessing2.csv'
dual_signature={path.name:hashlib.sha256(path.read_bytes()).hexdigest() for path in [dual_train_path,dual_test_path]}

# Reuse fixed features when several model notebooks run in the same kernel.
if globals().get('dual_cached_signature') != dual_signature:
    dual_train=pd.read_csv(dual_train_path,keep_default_na=False)
    dual_test=pd.read_csv(dual_test_path,keep_default_na=False)
    dual_names=dict(zip(dual_train.label_id,dual_train.label))
    assert sorted(dual_names)==list(range(12)) and dual_names[0]=='None'
    for dual_frame in [dual_train,dual_test]:
        assert (dual_frame.label_id.map(dual_names)==dual_frame.label).all()
        assert ((dual_frame.label=='None')==(dual_frame.is_toxic==0)).all()
    dual_vectorizers=[]
    dual_train_blocks=[]
    dual_test_blocks=[]
    for train_column,test_column in [('prev_clean','context_clean'),('target_clean','reply_clean')]:
        for analyzer,ngrams in [('word',(1,2)),('char_wb',(3,5))]:
            vectorizer=TfidfVectorizer(analyzer=analyzer,ngram_range=ngrams,min_df=2,max_features=50000,sublinear_tf=True)
            dual_train_blocks.append(vectorizer.fit_transform(dual_train[train_column]))
            dual_test_blocks.append(vectorizer.transform(dual_test[test_column]))
            dual_vectorizers.append(vectorizer)
    dual_X_train=hstack(dual_train_blocks,format='csr')
    dual_X_test=hstack(dual_test_blocks,format='csr')
    dual_cached_signature=dual_signature.copy()
print('Train / test rows:',len(dual_train),len(dual_test))
print('TF-IDF features:',dual_X_train.shape,dual_X_test.shape)


Train / test rows: 19431 1804
TF-IDF features: (19431, 38806) (1804, 38806)


In [2]:
def dual_make_model():
    return SVC(C=1.0,kernel='linear',class_weight='balanced',gamma='scale',cache_size=512,random_state=42)

dual_rows=[]
dual_predictions={}
dual_reports={}
dual_models={}
for dual_task,dual_column,dual_labels,dual_label_names in [
    ('binary','is_toxic',[0,1],['Non-toxic','Toxic']),
    ('subtype','label_id',list(range(12)),[dual_names[i] for i in range(12)]),
]:
    dual_model=dual_make_model()
    print('Training Linear-kernel SVC:',dual_task,flush=True)
    dual_y_train=dual_train[dual_column].to_numpy(dtype=np.int64)
    dual_y_test=dual_test[dual_column].to_numpy(dtype=np.int64)
    dual_model.fit(dual_X_train,dual_y_train)
    dual_pred=np.asarray(dual_model.predict(dual_X_test)).reshape(-1).astype(np.int64)
    dual_models[dual_task]=dual_model
    dual_predictions[dual_task]=dual_pred.tolist()
    dual_rows.append(dict(
        model='Linear-kernel SVC',task=dual_task,train_rows=len(dual_train),test_rows=len(dual_test),
        accuracy=float(accuracy_score(dual_y_test,dual_pred)),
        macro_f1=float(f1_score(dual_y_test,dual_pred,labels=dual_labels,average='macro',zero_division=0)),
        weighted_f1=float(f1_score(dual_y_test,dual_pred,labels=dual_labels,average='weighted',zero_division=0)),
    ))
    dual_reports[dual_task]=classification_report(dual_y_test,dual_pred,labels=dual_labels,target_names=dual_label_names,zero_division=0,output_dict=True)
    print(classification_report(dual_y_test,dual_pred,labels=dual_labels,target_names=dual_label_names,zero_division=0))
    # Save each completed task so an interrupted run keeps verified results.
    dual_checkpoint_path=dual_root/'HyperParamTune-TFIDF/SVC.ipynb'
    import json
    dual_checkpoint_document=json.loads(dual_checkpoint_path.read_text(encoding='utf-8'))
    dual_checkpoint_document['metadata']['dual_task_evaluation'].update(
        status='running',results=dual_rows,predictions=dual_predictions,
        classification_reports=dual_reports,input_sha256=dual_signature,
        feature_count=dual_X_train.shape[1],
    )
    dual_checkpoint_path.write_text(json.dumps(dual_checkpoint_document,ensure_ascii=False,indent=1),encoding='utf-8')
    print('Metrics:',dual_rows[-1],flush=True)
dual_results=pd.DataFrame(dual_rows)
display(dual_results.round(4))


[
  {
    "model": "Linear-kernel SVC",
    "task": "binary",
    "train_rows": 19431,
    "test_rows": 1804,
    "accuracy": 0.6042128603104213,
    "macro_f1": 0.577198602977863,
    "weighted_f1": 0.590942698813726
  }
]
Subtype status: binary_complete_subtype_interrupted_at_user_time_limit
